In [4]:
# !rm -rf sample_data/data/patch/
# !rm -rf sample_data/data/patch-binary/
# !rm -rf sample_data/data/patch-texture/

In [5]:
from google.colab import drive

# This will mount your entire Google Drive
drive.mount('/content/drive')

# After mounting, you can navigate to your specific folder
# Replace 'My Drive/path/to/your/folder' with the actual path to your folder
specific_folder_path = '/content/drive/MyDrive/disser/data'

# You can then list the contents of the folder to verify
import os
if os.path.exists(specific_folder_path):
  print(f"Contents of '{specific_folder_path}':")
  for item in os.listdir(specific_folder_path):
    print(item)
else:
  print(f"Folder not found at '{specific_folder_path}'")

ModuleNotFoundError: No module named 'google.colab'

In [1]:
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)

Sat Aug 16 18:55:22 2025       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.60.13    Driver Version: 525.60.13    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla V100-PCIE...  On   | 00000001:00:00.0 Off |                  Off |
| N/A   28C    P0    24W / 250W |    145MiB / 16384MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
                                                                               
+-------

In [2]:
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))

if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')

Your runtime has 118.2 gigabytes of available RAM

You are using a high-RAM runtime!


In [ ]:
import os
base_path = '/content/drive/MyDrive/disser/data' # Replace with your actual base path



# Use a loop to copy files from each folder
for i in range(1,13):
  source_tiles = os.path.join(base_path, f'tiles/{i}.jpg')
  source_texture = os.path.join(base_path, f'texture_tiles/{i}.png')
  source_binary = os.path.join(base_path, f'binary_tiles/{i}.png')
  destination_folder = '/content/sample_data/data'

  # Use the cp command with -f (force overwrite if needed) and -v (verbose output)
  !cp -f {source_tiles} {os.path.join(destination_folder, f'tiles')}
  !cp -f {source_binary} {os.path.join(destination_folder, f'binary_tiles')}
  !cp -fv {source_texture} {os.path.join(destination_folder, f'texture_tiles')}


# Verify the files have been copied (optional)
!ls /content/sample_data/data

'/content/drive/MyDrive/disser/data/texture_tiles/1.png' -> '/content/sample_data/data/texture_tiles/1.png'
'/content/drive/MyDrive/disser/data/texture_tiles/2.png' -> '/content/sample_data/data/texture_tiles/2.png'
'/content/drive/MyDrive/disser/data/texture_tiles/3.png' -> '/content/sample_data/data/texture_tiles/3.png'
'/content/drive/MyDrive/disser/data/texture_tiles/4.png' -> '/content/sample_data/data/texture_tiles/4.png'
'/content/drive/MyDrive/disser/data/texture_tiles/5.png' -> '/content/sample_data/data/texture_tiles/5.png'
'/content/drive/MyDrive/disser/data/texture_tiles/6.png' -> '/content/sample_data/data/texture_tiles/6.png'
'/content/drive/MyDrive/disser/data/texture_tiles/7.png' -> '/content/sample_data/data/texture_tiles/7.png'
'/content/drive/MyDrive/disser/data/texture_tiles/8.png' -> '/content/sample_data/data/texture_tiles/8.png'
'/content/drive/MyDrive/disser/data/texture_tiles/9.png' -> '/content/sample_data/data/texture_tiles/9.png'
'/content/drive/MyDrive/diss

In [ ]:
# # Create the destination directory if it doesn't exist
# !mkdir -p /content/sample_data/patch
# !mkdir -p /content/sample_data/data/patch-binary
# !mkdir -p /content/sample_data/data/patch-texture

# source_patch = os.path.join(base_path, f'patch')
# source_texture = os.path.join(base_path, f'patch-binary')
# source_binary = os.path.join(base_path, f'patch-texture')
# destination_folder = '/content/sample_data/data'

# ! cp -rf {source_patch} {os.path.join(destination_folder, f'patch')}
# ! cp -rf {source_binary} {os.path.join(destination_folder, f'patch-binary')}
# ! cp -rf {source_texture} {os.path.join(destination_folder, f'patch-texture')}


In [3]:
import os
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm
import torch
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from torchvision import models, transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.models import EfficientNet_B0_Weights
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
import glob
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
# Set random seed for reproducibility
# torch.manual_seed(42)
# np.random.seed(42)

# ======================== SETUP AND DATA LOADING ========================

def copy_files_from_drive(base_path):
    """Copy files from Google Drive to local directory"""
    destination_folder = '/content/sample_data/data'

    for i in range(1, 13):
        source_tiles = os.path.join(base_path, f'tiles/{i}.jpg')
        source_texture = os.path.join(base_path, f'texture_tiles/{i}.png')
        source_binary = os.path.join(base_path, f'binary_tiles/{i}.png')

        !cp -f {source_tiles} {os.path.join(destination_folder, 'tiles')}
        !cp -f {source_binary} {os.path.join(destination_folder, 'binary_tiles')}
        !cp -f {source_texture} {os.path.join(destination_folder, 'texture_tiles')}

    print("✅ Files copied successfully")

# ======================== TILE GENERATION ========================

def get_all_char_before_dot(s):
    """Helper function to extract filename without extension"""
    dot_index = s.find('.')
    return s[:dot_index] if dot_index > 0 else None

def generate_tiles(image_path, tile_width, tile_height, step, output_folder):
    """Generate tiles from large images"""
    image = Image.open(image_path)
    image_width, image_height = image.size

    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            bbox = (x, y, x + tile_width, y + tile_height)
            tile = image.crop(bbox)

            image_file_name = os.path.basename(image_path)
            image_idx = get_all_char_before_dot(image_file_name)
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png")

def generate_binary_tiles(image_path, tile_width, tile_height, step, output_folder):
    """Generate binary tiles from mask images"""
    image = Image.open(image_path).convert("L")
    image_width, image_height = image.size

    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            bbox = (x, y, x + tile_width, y + tile_height)
            tile = image.crop(bbox)
            tile = tile.point(lambda p: 255 if p > 127 else 0, mode="1")

            image_file_name = os.path.basename(image_path)
            image_idx = image_file_name.split('.')[0]
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png", format="PNG")

def generate_all_tiles(base_path):
    """Generate all tiles from original images"""
    # Generate tiles for original images
    for i in range(1, 13):
        generate_tiles(f'{base_path}/tiles/{i}.jpg', 256, 256, 128, f'{base_path}/patch')
    print("✅ Original tiles generated")

    # Generate binary tiles
    for i in range(1, 13):
        generate_binary_tiles(f'{base_path}/binary_tiles/{i}.png', 256, 256, 128, f'{base_path}/patch-binary')
    print("✅ Binary tiles generated")

    # Generate texture tiles
    for i in range(1, 13):
        generate_tiles(f'{base_path}/texture_tiles/{i}.png', 256, 256, 128, f'{base_path}/patch-texture')
    print("✅ Texture tiles generated")

# ======================== ENHANCED PATCH EXTRACTION ========================

def get_roof_texture_label(color_patch, min_pixel_threshold=50):
    """Enhanced function to extract roof texture label from color mask patch"""
    if len(color_patch.shape) == 3:
        image_rgb = color_patch
    else:
        image_rgb = cv2.cvtColor(color_patch, cv2.COLOR_BGR2RGB)

    red_channel = image_rgb[:, :, 0]
    green_channel = image_rgb[:, :, 1]
    blue_channel = image_rgb[:, :, 2]

    # Define thresholds
    high_threshold = 200
    low_threshold = 100

    # Count pixels with refined conditions
    rough_pixels = np.sum((red_channel > high_threshold) &
                         (green_channel < low_threshold) &
                         (blue_channel < low_threshold))

    smooth_pixels = np.sum((red_channel < low_threshold) &
                          (green_channel > high_threshold) &
                          (blue_channel < low_threshold))

    average_pixels = np.sum((red_channel > low_threshold) &
                           (red_channel < high_threshold) &
                           (green_channel > low_threshold) &
                           (green_channel < high_threshold) &
                           (blue_channel < low_threshold))

    pixel_counts = {
        "rough": rough_pixels,
        "smooth": smooth_pixels,
        "average": average_pixels
    }

    max_count = max(pixel_counts.values())

    if max_count < min_pixel_threshold:
        return "no_contour", pixel_counts
    else:
        label = max(pixel_counts.items(), key=lambda x: x[1])[0]
        return label, pixel_counts

def extract_multiple_rooftops_per_tile(image_paths, binary_paths, color_paths,
                                     min_contour_area=100, target_size=(224, 224)):
    """Extract multiple rooftops from each tile with contour mask storage"""
    patches = []
    labels = []
    tile_info = []

    for idx in tqdm(range(len(binary_paths)), desc="Extracting patches"):
        try:
            # Load images
            orig_img = np.array(Image.open(image_paths[idx]).convert("RGB")) / 255.0
            binary_mask = np.array(Image.open(binary_paths[idx]).convert("L")) / 255.0
            color_mask = np.array(Image.open(color_paths[idx]).convert("RGB")) / 255.0

            # Convert binary mask for contour detection
            binary_cv = (binary_mask * 255).astype(np.uint8)

            # Find ALL contours
            contours, _ = cv2.findContours(binary_cv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Filter and sort contours
            valid_contours = [c for c in contours if cv2.contourArea(c) > min_contour_area]

            if valid_contours:
                valid_contours.sort(key=cv2.contourArea, reverse=True)

                for contour_idx, contour in enumerate(valid_contours):
                    x, y, w, h = cv2.boundingRect(contour)

                    # Add padding
                    padding = 10
                    x = max(0, x - padding)
                    y = max(0, y - padding)
                    w = min(orig_img.shape[1] - x, w + 2*padding)
                    h = min(orig_img.shape[0] - y, h + 2*padding)

                    # Extract patches
                    orig_patch = orig_img[y:y+h, x:x+w]
                    color_patch = color_mask[y:y+h, x:x+w]

                    # Skip if too small
                    if orig_patch.shape[0] < 20 or orig_patch.shape[1] < 20:
                        continue

                    # Create contour mask for this specific patch
                    contour_mask = np.zeros((orig_img.shape[0], orig_img.shape[1]), dtype=np.uint8)
                    cv2.fillPoly(contour_mask, [contour], 255)

                    # Extract the contour mask for this patch area
                    patch_contour_mask = contour_mask[y:y+h, x:x+w]

                    # Resize patches
                    orig_patch_resized = cv2.resize(orig_patch, target_size, interpolation=cv2.INTER_AREA)
                    color_patch_resized = cv2.resize(color_patch, target_size, interpolation=cv2.INTER_AREA)

                    # Get label
                    label, pixel_counts = get_roof_texture_label(
                        (color_patch_resized * 255).astype(np.uint8),
                        min_pixel_threshold=50
                    )

                    patches.append(orig_patch_resized)
                    labels.append(label)

                    # Store info with contour information
                    tile_info.append({
                        'tile_idx': idx,
                        'contour_idx': contour_idx,
                        'contour_area': cv2.contourArea(contour),
                        'bbox': (x, y, w, h),
                        'pixel_counts': pixel_counts,
                        'confidence': max(pixel_counts.values()) / sum(pixel_counts.values()) if sum(pixel_counts.values()) > 0 else 0,
                        'contour_mask': patch_contour_mask,  # Store contour mask
                        'original_contour': contour  # Store original contour
                    })
            else:
                # No valid contours
                orig_patch = cv2.resize(orig_img, target_size, interpolation=cv2.INTER_AREA)
                label = "no_contour"

                patches.append(orig_patch)
                labels.append(label)
                tile_info.append({
                    'tile_idx': idx,
                    'contour_idx': -1,
                    'contour_area': 0,
                    'bbox': (0, 0, orig_img.shape[1], orig_img.shape[0]),
                    'pixel_counts': {'rough': 0, 'smooth': 0, 'average': 0},
                    'confidence': 0
                })

        except Exception as e:
            print(f"Error processing tile {idx}: {str(e)}")
            continue

    return np.array(patches), labels, tile_info

def load_tiled_data(original_dir, binary_dir, color_dir):
    """Load paths for tiled data"""
    filenames = sorted(os.listdir(original_dir))
    image_paths, binary_paths, color_paths = [], [], []

    for filename in tqdm(filenames, desc="Loading tile paths"):
        img_path = os.path.join(original_dir, filename)
        binary_path = os.path.join(binary_dir, filename)
        color_path = os.path.join(color_dir, filename)

        if os.path.exists(img_path) and os.path.exists(binary_path) and os.path.exists(color_path):
            image_paths.append(img_path)
            binary_paths.append(binary_path)
            color_paths.append(color_path)

    return image_paths, binary_paths, color_paths

# ======================== DATA ANALYSIS AND FILTERING ========================

def analyze_data_distribution(labels, tile_info):
    """Analyze the distribution of extracted data"""
    print("\n=== DATA DISTRIBUTION ANALYSIS ===")

    # Count labels
    label_counts = Counter(labels)
    print(f"📊 Total patches extracted: {len(labels)}")
    print("Label distribution:")
    for label, count in label_counts.items():
        print(f"  {label}: {count} ({count/len(labels)*100:.1f}%)")

    # Analyze confidence scores
    confidences = [info['confidence'] for info in tile_info]
    print(f"\n🎯 Confidence scores:")
    print(f"  Mean: {np.mean(confidences):.3f}")
    print(f"  Median: {np.median(confidences):.3f}")
    print(f"  Min: {np.min(confidences):.3f}")
    print(f"  Max: {np.max(confidences):.3f}")

    # Count patches per tile
    tiles_with_patches = Counter([info['tile_idx'] for info in tile_info])
    patches_per_tile = list(tiles_with_patches.values())
    print(f"\n🔢 Patches per tile:")
    print(f"  Mean: {np.mean(patches_per_tile):.1f}")
    print(f"  Median: {np.median(patches_per_tile):.1f}")
    print(f"  Max: {np.max(patches_per_tile)}")

    return label_counts, confidences, patches_per_tile

def filter_low_confidence_samples(patches, labels, tile_info, min_confidence=0.3):
    """Filter out low confidence samples"""
    print(f"\n🔍 Filtering samples with confidence < {min_confidence}")

    filtered_patches = []
    filtered_labels = []
    filtered_info = []

    for patch, label, info in zip(patches, labels, tile_info):
        if info['confidence'] >= min_confidence or label == "no_contour":
            filtered_patches.append(patch)
            filtered_labels.append(label)
            filtered_info.append(info)

    print(f"✅ Kept {len(filtered_patches)} out of {len(patches)} samples")
    return np.array(filtered_patches), filtered_labels, filtered_info

def visualize_extracted_patches(patches, labels, tile_info, num_samples=16):
    """Visualize sample patches"""
    indices = np.random.choice(len(patches), min(num_samples, len(patches)), replace=False)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        patch = patches[idx]
        label = labels[idx]
        confidence = tile_info[idx]['confidence']

        axes[i].imshow(patch)
        axes[i].set_title(f'{label}\nConf: {confidence:.2f}', fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# ======================== MODEL TRAINING SETUP ========================

def compute_class_weights(labels):
    """Compute class weights for balanced training"""
    unique_labels = list(set(labels))
    label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
    numerical_labels = [label_to_idx[label] for label in labels]

    class_weights = compute_class_weight('balanced',
                                       classes=np.unique(numerical_labels),
                                       y=numerical_labels)

    print("\n⚖️ Class weights:")
    for label, idx in label_to_idx.items():
        print(f"  {label}: {class_weights[idx]:.3f}")

    return torch.FloatTensor(class_weights), label_to_idx

def create_data_loaders(patches, labels, label_to_idx, batch_size=32, test_size=0.2):
    """Create balanced data loaders with progress tracking"""
    print("🔄 Creating data loaders...")

    # Data transformations
    transform_train = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    transform_val = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Convert labels to numerical
    print("📊 Converting labels to numerical format...")
    numerical_labels = [label_to_idx[label] for label in tqdm(labels, desc="Converting labels")]

    # Apply transforms
    X_train_transformed = []
    X_val_transformed = []

    # Convert to tensors first
    print("🔄 Converting patches to tensors...")
    X = torch.stack([torch.from_numpy(patch.transpose(2, 0, 1)).float()
                     for patch in tqdm(patches, desc="Converting to tensors")])
    y = torch.LongTensor(numerical_labels)

    # Stratified split
    print("✂️ Splitting data into train/validation sets...")
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    print(f"📊 Train samples: {len(X_train)}, Validation samples: {len(X_val)}")

    # Apply transforms after split
    print("🎨 Applying training transforms...")
    for patch in tqdm(X_train, desc="Applying train transforms"):
        patch_img = patch.permute(1, 2, 0).numpy()
        X_train_transformed.append(transform_train(patch_img))

    print("🎨 Applying validation transforms...")
    for patch in tqdm(X_val, desc="Applying val transforms"):
        patch_img = patch.permute(1, 2, 0).numpy()
        X_val_transformed.append(transform_val(patch_img))

    print("📦 Stacking transformed tensors...")
    X_train = torch.stack(X_train_transformed)
    X_val = torch.stack(X_val_transformed)

    # Create weighted sampler for training
    print("⚖️ Computing class weights for balanced sampling...")
    class_sample_count = np.array([len(np.where(y_train == t)[0]) for t in np.unique(y_train)])
    weight = 1. / class_sample_count
    samples_weight = np.array([weight[t] for t in y_train])
    samples_weight = torch.from_numpy(samples_weight).double()
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

    # Create datasets and loaders
    print("🗂️ Creating datasets and data loaders...")
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    print(f"✅ Data loaders created successfully!")
    print(f"   📊 Training batches: {len(train_loader)}")
    print(f"   📊 Validation batches: {len(val_loader)}")
    print(f"   📦 Batch size: {batch_size}")

    return train_loader, val_loader


def create_model(num_classes, class_weights, device):
    """Create and setup the model"""
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    model = model.to(device)

    # Weighted loss function
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    return model, criterion, optimizer, scheduler

# ======================== TRAINING LOOP ========================

def train_model(model, criterion, optimizer, scheduler, train_loader, val_loader,
                num_epochs=100, patience=10, checkpoint_dir="data/checkpoints"):
    """Train the model with early stopping"""

    device = next(model.parameters()).device
    best_val_loss = float('inf')
    trigger_times = 0
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    print(f"\n🚀 Starting training for {num_epochs} epochs")
    print(f"📱 Using device: {device}")

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for inputs, targets in train_bar:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += targets.size(0)
            correct_train += (predicted == targets).sum().item()

            train_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })

        epoch_train_loss = running_loss / len(train_loader.dataset)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(epoch_train_loss)
        train_accuracies.append(train_accuracy)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        all_preds, all_targets = [], []

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for inputs, targets in val_bar:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total_val += targets.size(0)
                correct_val += (predicted == targets).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())

                val_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * correct_val / total_val:.2f}%'
                })

        epoch_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = 100 * correct_val / total_val
        val_losses.append(epoch_val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch {epoch+1}: Train Loss: {epoch_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")

        # Save checkpoint
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': epoch_train_loss,
            'val_loss': epoch_val_loss,
            'train_accuracy': train_accuracy,
            'val_accuracy': val_accuracy,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accuracies': train_accuracies,
            'val_accuracies': val_accuracies
        }, checkpoint_path)

        # Learning rate scheduling
        scheduler.step(epoch_val_loss)

        # Early stopping
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            trigger_times = 0
            torch.save(model.state_dict(), f"{checkpoint_dir}/best_texture_model.pth")
            print(f"✅ New best model saved with val_loss: {best_val_loss:.4f}")
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f"🛑 Early stopping triggered after {patience} epochs without improvement!")
                break

    return train_losses, val_losses, train_accuracies, val_accuracies, all_preds, all_targets

# ======================== EVALUATION AND VISUALIZATION ========================

def evaluate_model(all_targets, all_preds, label_to_idx):
    """Evaluate model performance"""
    # Reverse label mapping
    idx_to_label = {v: k for k, v in label_to_idx.items()}

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='weighted')

    print(f"\n📊 FINAL RESULTS:")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

    # Classification report
    print(f"\n📋 Classification Report:")
    target_names = [idx_to_label[i] for i in sorted(idx_to_label.keys())]
    print(classification_report(all_targets, all_preds, target_names=target_names))

    return precision, recall, f1

def plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training history"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    ax1.plot(train_losses, label="Train Loss", color='blue')
    ax1.plot(val_losses, label="Val Loss", color='red')
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss Over Epochs")
    ax1.legend()
    ax1.grid(True)

    # Accuracy plot
    ax2.plot(train_accuracies, label="Train Acc", color='blue')
    ax2.plot(val_accuracies, label="Val Acc", color='red')
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("Accuracy Over Epochs")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(all_targets, all_preds, label_to_idx):
    """Plot confusion matrix"""
    idx_to_label = {v: k for k, v in label_to_idx.items()}
    labels = [idx_to_label[i] for i in sorted(idx_to_label.keys())]
    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(10, 8))
    im = plt.imshow(cm, interpolation='nearest', cmap='Blues')
    plt.colorbar(im)

    # Add text annotations
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], 'd'),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")

    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.xticks(range(len(labels)), labels, rotation=45)
    plt.yticks(range(len(labels)), labels)
    plt.tight_layout()
    plt.show()

In [4]:
base_path = "/home/student/sky-scan/data"
def run_complete_training_pipeline():
    """
    Complete pipeline that loads images, extracts patches, trains model with checkpointing
    """
    print("🚀 Starting Complete Training Pipeline")
    print("=" * 60)

    # ======================== SETUP ========================

    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"📱 Using device: {device}")

    # Create directories
    os.makedirs(f'{base_path}/patch', exist_ok=True)
    os.makedirs(f'{base_path}/patch-binary', exist_ok=True)
    os.makedirs(f'{base_path}//patch-texture', exist_ok=True)
    os.makedirs(f'{base_path}//checkpoints', exist_ok=True)

    checkpoint_dir = f'{base_path}/checkpoints'

    print("✅ Directories created")

    # ======================== GENERATE TILES ========================

    print("\n🔧 Generating tiles from original images...")
    try:
        generate_all_tiles(base_path)
        print("✅ All tiles generated successfully")
    except Exception as e:
        print(f"❌ Error generating tiles: {e}")
        return None

    # ======================== LOAD DATA PATHS ========================

    print("\n📂 Loading image paths...")
    try:
        image_paths, binary_paths, color_paths = load_tiled_data(
            f'{base_path}/patch',
            f'{base_path}/patch-binary',
            f'{base_path}/patch-texture'
        )
        print(f"✅ Loaded {len(image_paths)} image sets")
    except Exception as e:
        print(f"❌ Error loading data paths: {e}")
        return None

    # ======================== EXTRACT PATCHES ========================

    print("\n🔍 Extracting rooftop patches...")
    try:
        patches, labels, tile_info = extract_multiple_rooftops_per_tile(
            image_paths, binary_paths, color_paths,
            min_contour_area=100,
            target_size=(224, 224)
        )
        print(f"✅ Extracted {len(patches)} patches")
    except Exception as e:
        print(f"❌ Error extracting patches: {e}")
        return None

    # ======================== ANALYZE AND FILTER DATA ========================

    print("\n📊 Analyzing data distribution...")
    label_counts, confidences, patches_per_tile = analyze_data_distribution(labels, tile_info)

    # Filter low confidence samples
    patches, labels, tile_info = filter_low_confidence_samples(
        patches, labels, tile_info, min_confidence=0.3
    )

    # Visualize sample patches
    print("\n🖼️ Visualizing sample patches...")
    visualize_extracted_patches(patches, labels, tile_info, num_samples=16)

    # ======================== PREPARE TRAINING DATA ========================

    print("\n⚖️ Computing class weights and preparing data...")
    class_weights, label_to_idx = compute_class_weights(labels)

    # Create data loaders
    train_loader, val_loader = create_data_loaders(
        patches, labels, label_to_idx,
        batch_size=32, test_size=0.2
    )

    num_classes = len(label_to_idx)
    print(f"📊 Number of classes: {num_classes}")
    print(f"📋 Classes: {list(label_to_idx.keys())}")

    # ======================== CREATE MODEL ========================

    print("\n🤖 Creating model...")
    model, criterion, optimizer, scheduler = create_model(
        num_classes, class_weights, device
    )

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"📊 Total parameters: {total_params:,}")
    print(f"📊 Trainable parameters: {trainable_params:,}")

    # ======================== TRAIN MODEL ========================

    print("\n🎯 Starting model training...")
    try:
        train_losses, val_losses, train_accuracies, val_accuracies, all_preds, all_targets = train_model(
            model, criterion, optimizer, scheduler,
            train_loader, val_loader,
            num_epochs=100,
            patience=10,
            checkpoint_dir=checkpoint_dir
        )
        print("✅ Training completed successfully!")
    except Exception as e:
        print(f"❌ Error during training: {e}")
        return None

    # ======================== EVALUATE MODEL ========================

    print("\n📈 Evaluating model performance...")
    precision, recall, f1 = evaluate_model(all_targets, all_preds, label_to_idx)

    # Plot training history
    print("\n📊 Plotting training history...")
    plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies)

    # Plot confusion matrix
    print("\n🔍 Plotting confusion matrix...")
    plot_confusion_matrix(all_targets, all_preds, label_to_idx)

    # ======================== SAVE FINAL RESULTS ========================

    print("\n💾 Saving final results...")

    # Save training history
    training_history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'final_metrics': {
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        },
        'label_to_idx': label_to_idx,
        'class_weights': class_weights.tolist(),
        'num_patches': len(patches),
        'num_classes': num_classes
    }

    import json
    with open(f'{checkpoint_dir}/training_history.json', 'w') as f:
        json.dump(training_history, f, indent=2)

    # Save model info
    model_info = {
        'model_type': 'EfficientNet-B0',
        'num_classes': num_classes,
        'classes': list(label_to_idx.keys()),
        'input_size': (224, 224),
        'total_parameters': total_params,
        'trainable_parameters': trainable_params,
        'training_completed': True
    }

    with open(f'{checkpoint_dir}/model_info.json', 'w') as f:
        json.dump(model_info, f, indent=2)

    print("✅ Training history and model info saved")
    print(f"📁 Checkpoints saved in: {checkpoint_dir}")
    print(f"🏆 Best model saved as: {checkpoint_dir}/best_texture_model.pth")

    # ======================== SUMMARY ========================

    print("\n" + "=" * 60)
    print("🎉 TRAINING PIPELINE COMPLETED SUCCESSFULLY!")
    print("=" * 60)
    print(f"📊 Final Results:")
    print(f"   • Total patches processed: {len(patches)}")
    print(f"   • Number of classes: {num_classes}")
    print(f"   • Final F1-Score: {f1:.4f}")
    print(f"   • Final Precision: {precision:.4f}")
    print(f"   • Final Recall: {recall:.4f}")
    print(f"   • Best model saved: ✅")
    print(f"   • Checkpoints saved: ✅")
    print("=" * 60)

    return {
        'model': model,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies,
        'label_to_idx': label_to_idx,
        'checkpoint_dir': checkpoint_dir,
        'final_metrics': {'precision': precision, 'recall': recall, 'f1_score': f1}
    }

# ======================== RUN THE COMPLETE PIPELINE ========================

# Execute the complete training pipeline
print("🏁 Starting the complete training pipeline...")
results = run_complete_training_pipeline()

if results is not None:
    print("\n🎯 Pipeline execution completed successfully!")
    print("📂 All files saved and model training completed.")
    print("\n📋 Saved files:")
    print("   • Best model: sample_data/data/checkpoints/best_texture_model.pth")
    print("   • Training history: sample_data/data/checkpoints/training_history.json")
    print("   • Model info: sample_data/data/checkpoints/model_info.json")
    print("   • Epoch checkpoints: sample_data/data/checkpoints/checkpoint_epoch_*.pth")
else:
    print("\n❌ Pipeline execution failed. Please check the error messages above.")


🏁 Starting the complete training pipeline...
🚀 Starting Complete Training Pipeline
📱 Using device: cuda
✅ Directories created

🔧 Generating tiles from original images...


KeyboardInterrupt: 

Import all libraries

In [1]:
import os
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
from torchvision import models, transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.models import EfficientNet_B0_Weights
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
import glob
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter


In [14]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Define paths to your tiled data
# base_path = "/content/sample_data/data"

base_path = "/home/student/sky-scan/data"

original_dir = f"{base_path}/patch"
binary_dir = f"{base_path}/patch-binary"
color_dir = f"{base_path}/patch-texture"
images_, binary_masks_, color_masks_ = [], [], []


Generate tiles for binary and colour images

In [15]:
import os
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm
import torch
from torch.utils.data import TensorDataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
import matplotlib.pyplot as plt
from torchvision import models, transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.models import EfficientNet_B0_Weights
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import cv2
import matplotlib.pyplot as plt
import glob
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter
import gc

def get_all_char_before_dot(s):
    dot_index = s.find('.')
    if dot_index > 0:
        return s[:dot_index]
    else:
        return None

def generate_tiles(image_path, tile_width, tile_height, step, output_folder):
    # Open the image
    image = Image.open(image_path)
    image_width, image_height = image.size

    # Generate tiles
    image_file_name=os.path.basename(image_path)
    image_idx = get_all_char_before_dot(image_file_name)
    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            # Define the bounding box for the current tile
            left = x
            upper = y
            right = left + tile_width
            lower = upper + tile_height
            bbox = (left, upper, right, lower)

            # Crop the image to the bounding box to create the tile
            tile = image.crop(bbox)

            # Save the tile to the output folder
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png") #image_id_x_y.png

    print(f"✅ Tiles from {image_file_name} are generated successfully.")
    image.close()
    del image
    gc.collect()


def generate_binary_tiles(image_path, tile_width, tile_height, step, output_folder):
    # Open the image in grayscale mode (black & white)
    image = Image.open(image_path).convert("L")  # "L" mode ensures grayscale (0-255)
    image_width, image_height = image.size

    # Generate tiles
    image_file_name = os.path.basename(image_path)
    image_idx = image_file_name.split('.')[0]  # Get filename without extension
    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            # Define the bounding box for the current tile
            left = x
            upper = y
            right = left + tile_width
            lower = upper + tile_height
            bbox = (left, upper, right, lower)

            # Crop the image to create the tile
            tile = image.crop(bbox)

            # Convert to binary (0 or 255) to ensure black and white format
            tile = tile.point(lambda p: 255 if p > 127 else 0, mode="1")  # Thresholding

            # Save the tile to the output folder
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png", format="PNG")  # Save as binary PNG

    print(f"✅ Binary tiles from {image_file_name} are generated successfully.")
    image.close()
    del image
    gc.collect()


# # Generate tiles for original images
# os.makedirs(f'{base_path}/patch', exist_ok=True)
#
# for i in range(1,2): #12 images
#     generate_tiles(f'{base_path}/tiles/{i}.jpg', 256, 256, 128, f'{base_path}/patch')


# print("successfully generated Patch tiles from original images.")

# # Generate tiles for binary mask images
# os.makedirs(f'{base_path}/patch-binary', exist_ok=True)

# for i in range(1,13): #12 images
#     generate_binary_tiles(f'{base_path}/binary_tiles/{i}.png', 256, 256, 128, f'{base_path}/patch-binary')

# print("Binary mask images are generated successfully.")

# # Generate tiles for original images
# os.makedirs(f'{base_path}/patch-texture', exist_ok=True)

# for i in range(1,13): #12 mages
#     generate_tiles(f'{base_path}/texture_tiles/{i}.png', 256, 256, 128, f'{base_path}/patch-texture')

Load binary and colour images for classification

In [16]:
def load_tiled_data(original_dir, binary_dir, color_dir):
    filenames = sorted(os.listdir(original_dir))
    image_paths = []
    binary_paths = []
    color_paths = []
    for filename in tqdm(filenames, desc="Loading tile paths", unit="tile"):
        img_path = os.path.join(original_dir, filename)
        binary_path = os.path.join(binary_dir, filename)
        color_path = os.path.join(color_dir, filename)
        if os.path.exists(img_path) and os.path.exists(binary_path) and os.path.exists(color_path):
            image_paths.append(img_path)
            binary_paths.append(binary_path)
            color_paths.append(color_path)
        else:
            print(f"Skipping file {filename}: Corresponding binary or color mask not found.")

    return image_paths, binary_paths, color_paths

image_paths, binary_paths, color_paths = load_tiled_data(original_dir, binary_dir, color_dir)

Loading tile paths: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16200/16200 [00:00<00:00, 21676.74tile/s]


In [17]:
print("len of images path :", len(image_paths))
print("len of binary images path :", len(binary_paths))
print("len of colour images path :", len(color_paths))

len of images path : 16200
len of binary images path : 16200
len of colour images path : 16200


helper function to extract label from tiles version of the image for deep leanring

In [18]:
def get_roof_texture_label(color_patch, min_pixel_threshold=50):
    """
    Enhanced function to extract roof texture label from color mask patch
    """
    # Convert to RGB if needed
    if len(color_patch.shape) == 3:
        image_rgb = color_patch
    else:
        image_rgb = cv2.cvtColor(color_patch, cv2.COLOR_BGR2RGB)

    # Get color channels
    red_channel = image_rgb[:, :, 0]
    green_channel = image_rgb[:, :, 1]
    blue_channel = image_rgb[:, :, 2]

    # Define thresholds for color detection
    high_threshold = 200
    low_threshold = 100

    # Count pixels for each category with more refined conditions
    rough_pixels = np.sum((red_channel > high_threshold) &
                         (green_channel < low_threshold) &
                         (blue_channel < low_threshold))

    smooth_pixels = np.sum((red_channel < low_threshold) &
                          (green_channel > high_threshold) &
                          (blue_channel < low_threshold))

    average_pixels = np.sum((red_channel > low_threshold) &
                           (red_channel < high_threshold) &
                           (green_channel > low_threshold) &
                           (green_channel < high_threshold) &
                           (blue_channel < low_threshold))

    # Create dictionary of pixel counts
    pixel_counts = {
        "rough": rough_pixels,
        "smooth": smooth_pixels,
        "average": average_pixels
    }

    # Get label with highest pixel count
    max_count = max(pixel_counts.values())

    if max_count < min_pixel_threshold:
        return "no_contour", pixel_counts
    else:
        label = max(pixel_counts.items(), key=lambda x: x[1])[0]
        return label, pixel_counts

def extract_multiple_rooftops_per_tile(image_paths, binary_paths, color_paths,
                                     min_contour_area=100, target_size=(224, 224)):
    """
    Enhanced function to extract multiple rooftops from each tile
    """
    patches = []
    labels = []
    tile_info = []  # Store information about each patch

    for idx in tqdm(range(len(binary_paths)), desc="Extracting patches"):
        try:
            # Load images from paths
            orig_img = np.array(Image.open(image_paths[idx]).convert("RGB")) / 255.0
            binary_mask = np.array(Image.open(binary_paths[idx]).convert("L")) / 255.0
            color_mask = np.array(Image.open(color_paths[idx]).convert("RGB")) / 255.0

            # Convert binary mask for contour detection
            binary_cv = (binary_mask * 255).astype(np.uint8)

            # Find ALL contours in binary mask
            contours, _ = cv2.findContours(binary_cv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Filter contours by area and extract patches for each
            valid_contours = [c for c in contours if cv2.contourArea(c) > min_contour_area]

            if valid_contours:
                # Sort contours by area (largest first)
                valid_contours.sort(key=cv2.contourArea, reverse=True)

                for contour_idx, contour in enumerate(valid_contours):
                    x, y, w, h = cv2.boundingRect(contour)

                    # Add padding to ensure we capture the full rooftop
                    padding = 10
                    x = max(0, x - padding)
                    y = max(0, y - padding)
                    w = min(orig_img.shape[1] - x, w + 2*padding)
                    h = min(orig_img.shape[0] - y, h + 2*padding)

                    # Extract patches from all images using same coordinates
                    orig_patch = orig_img[y:y+h, x:x+w]
                    color_patch = color_mask[y:y+h, x:x+w]

                    # Skip if patch is too small
                    if orig_patch.shape[0] < 20 or orig_patch.shape[1] < 20:
                        continue

                    # Resize patches
                    orig_patch_resized = cv2.resize(orig_patch, target_size,
                                                  interpolation=cv2.INTER_AREA)
                    color_patch_resized = cv2.resize(color_patch, target_size,
                                                   interpolation=cv2.INTER_AREA)

                    # Get label from color patch
                    label, pixel_counts = get_roof_texture_label(
                        (color_patch_resized * 255).astype(np.uint8),
                        min_pixel_threshold=50
                    )

                    # Store patch and label
                    patches.append(orig_patch_resized)
                    labels.append(label)

                    # Store additional information
                    tile_info.append({
                        'tile_idx': idx,
                        'contour_idx': contour_idx,
                        'contour_area': cv2.contourArea(contour),
                        'bbox': (x, y, w, h),
                        'pixel_counts': pixel_counts,
                        'confidence': max(pixel_counts.values()) / sum(pixel_counts.values()) if sum(pixel_counts.values()) > 0 else 0
                    })
            else:
                # If no valid contours, use full image but mark as low confidence
                orig_patch = cv2.resize(orig_img, target_size, interpolation=cv2.INTER_AREA)
                label = "no_contour"

                patches.append(orig_patch)
                labels.append(label)
                tile_info.append({
                    'tile_idx': idx,
                    'contour_idx': -1,
                    'contour_area': 0,
                    'bbox': (0, 0, orig_img.shape[1], orig_img.shape[0]),
                    'pixel_counts': {'rough': 0, 'smooth': 0, 'average': 0},
                    'confidence': 0
                })

        except Exception as e:
            print(f"Error processing tile {idx}: {str(e)}")
            continue
        finally:
            # Explicitly close images and collect garbage
            try:
                Image.open(image_paths[idx]).close()
                Image.open(binary_paths[idx]).close()
                Image.open(color_paths[idx]).close()
            except Exception as cleanup_e:
                print(f"Error during image cleanup for tile {idx}: {str(cleanup_e)}")
            del orig_img, binary_mask, color_mask
            gc.collect()


    return np.array(patches), labels, tile_info

def analyze_data_distribution(labels, tile_info):
    """
    Analyze the distribution of extracted data
    """
    print("=== Data Distribution Analysis ===")

    # Count labels
    label_counts = Counter(labels)
    print(f"Total patches extracted: {len(labels)}")
    print("Label distribution:")
    for label, count in label_counts.items():
        print(f"  {label}: {count} ({count/len(labels)*100:.1f}%)")

    # Analyze confidence scores
    confidences = [info['confidence'] for info in tile_info]
    print(f"\nConfidence scores:")
    print(f"  Mean: {np.mean(confidences):.3f}")
    print(f"  Median: {np.median(confidences):.3f}")
    print(f"  Min: {np.min(confidences):.3f}")
    print(f"  Max: {np.max(confidences):.3f}")

    # Count patches per tile
    tiles_with_patches = Counter([info['tile_idx'] for info in tile_info])
    patches_per_tile = list(tiles_with_patches.values())
    print(f"\nPatches per tile:")
    print(f"  Mean: {np.mean(patches_per_tile):.1f}")
    print(f"  Median: {np.median(patches_per_tile):.1f}")
    print(f"  Max: {np.max(patches_per_tile)}")

    return label_counts, confidences, patches_per_tile

def filter_low_confidence_samples(patches, labels, tile_info, min_confidence=0.3):
    """
    Filter out low confidence samples to improve training data quality
    """
    print(f"Filtering samples with confidence < {min_confidence}")

    filtered_patches = []
    filtered_labels = []
    filtered_info = []

    for patch, label, info in zip(patches, labels, tile_info):
        if info['confidence'] >= min_confidence or label == "no_contour":
            filtered_patches.append(patch)
            filtered_labels.append(label)
            filtered_info.append(info)

    print(f"Kept {len(filtered_patches)} out of {len(patches)} samples")
    return np.array(filtered_patches), filtered_labels, filtered_info

def compute_class_weights(labels):
    """
    Compute class weights to handle class imbalance
    """
    unique_labels = list(set(labels))
    label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
    numerical_labels = [label_to_idx[label] for label in labels]

    class_weights = compute_class_weight('balanced',
                                       classes=np.unique(numerical_labels),
                                       y=numerical_labels)

    weight_dict = {i: weight for i, weight in enumerate(class_weights)}
    print("Class weights:")
    for label, idx in label_to_idx.items():
        print(f"  {label}: {weight_dict[idx]:.3f}")

    return torch.FloatTensor(class_weights)

def visualize_extracted_patches(patches, labels, tile_info, num_samples=16):
    """
    Visualize a sample of extracted patches with their labels and confidence
    """
    indices = np.random.choice(len(patches), min(num_samples, len(patches)), replace=False)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        patch = patches[idx]
        label = labels[idx]
        confidence = tile_info[idx]['confidence']

        axes[i].imshow(patch)
        axes[i].set_title(f'{label}\nConf: {confidence:.2f}', fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

def create_weighted_loss_function(class_weights, device):
    """
    Create a weighted cross-entropy loss function
    """
    class_weights = class_weights.to(device)
    return torch.nn.CrossEntropyLoss(weight=class_weights)

def create_balanced_data_loader(patches, labels, batch_size=32, test_size=0.2):
    """
    Create balanced data loaders with stratified sampling
    """
    from sklearn.model_selection import train_test_split
    from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler

    # Convert labels to numerical
    label_map = {'rough': 0, 'average': 1, 'smooth': 2, 'no_contour': 3}
    numerical_labels = [label_map[label] for label in labels]

    # Convert to tensors
    X = torch.stack([transform_train(patch) for patch in patches])
    y = torch.LongTensor(numerical_labels)

    # Stratified split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    # Create weighted sampler for training data
    class_sample_count = np.array([len(np.where(y_train == t)[0]) for t in np.unique(y_train)])
    weight = 1. / class_sample_count
    samples_weight = np.array([weight[t] for t in y_train])
    samples_weight = torch.from_numpy(samples_weight).double()
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

    # Create datasets and loaders
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, label_map

In [19]:
patches, labels, tile_info = extract_multiple_rooftops_per_tile(
        image_paths, binary_paths, color_paths,
        min_contour_area=100,
        target_size=(224, 224)
    )
print("=== Data Extraction Complete ===")
print(f"Total patches extracted: {len(patches)}")

Extracting patches:  90%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊               | 14566/16200 [39:25<04:25,  6.16it/s]

Error processing tile 14566: cannot identify image file '/home/student/sky-scan/data/patch/8_4992_2688.png'
Error during image cleanup for tile 14566: cannot identify image file '/home/student/sky-scan/data/patch/8_4992_2688.png'


UnboundLocalError: local variable 'orig_img' referenced before assignment

In [20]:
# Analyze data distribution
label_counts, confidences, patches_per_tile = analyze_data_distribution(labels, tile_info)

# Filter low confidence samples
filtered_patches, filtered_labels, filtered_info = filter_low_confidence_samples(
    patches, labels, tile_info, min_confidence=0.3
)

# Compute class weights for handling imbalance
class_weights = compute_class_weights(filtered_labels)

# Release memory
del patches, labels, tile_info
gc.collect()

NameError: name 'labels' is not defined

show random number of sample to make sure everything is setup correct

In [ ]:
# Visualize some samples
print("\nVisualizing sample patches...")
visualize_extracted_patches(filtered_patches, filtered_labels, filtered_info)

In [ ]:
# def display_samples_with_ground_truth(patches, labels, color_masks, num_samples=30):
#     # Select random indices
#     indices = np.random.choice(len(patches), num_samples, replace=False)

#     # Create subplot grid (num_samples rows, 4 columns)
#     fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))

#     for idx, sample_idx in enumerate(indices):
#         # Get patch and its label
#         patch = patches[sample_idx]

#         # Get corresponding color mask
#         color_mask = color_masks[sample_idx]

#         # Convert binary mask for contour detection
#         binary_cv = (binary_masks_[sample_idx] * 255).astype(np.uint8)
#         # Find contours in binary mask and get largest contour
#         contours, _ = cv2.findContours(binary_cv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
#         if contours:
#             largest_contour = max(contours, key=cv2.contourArea)
#             x, y, w, h = cv2.boundingRect(largest_contour)
#             color_patch = cv2.resize(color_mask[y:y+h, x:x+w], (20, 20))
#         else:
#             color_patch = cv2.resize(color_mask, (20, 20))

#         # Get ground truth label from color mask
#         mask_rgb_cv = (color_mask * 255).astype(np.uint8)
#         _, label = get_largest_contour_square(
#             cv2.cvtColor(mask_rgb_cv, cv2.COLOR_RGB2BGR),
#             black_tolerance=10
#         )

#         # Display original patch
#         axes[idx, 0].imshow(patch)
#         axes[idx, 0].set_title('Original Patch')
#         axes[idx, 0].axis('off')

#         # Display full color mask
#         axes[idx, 1].imshow(color_mask)
#         axes[idx, 1].set_title('Full Color Mask')
#         axes[idx, 1].axis('off')

#         # Display cut color mask patch
#         axes[idx, 2].imshow(color_patch)
#         axes[idx, 2].set_title('Cut Color Mask')
#         axes[idx, 2].axis('off')

#         # Display labels
#         axes[idx, 3].text(0.5, 0.5, f'Calculated: {label}',
#                          horizontalalignment='center',
#                          verticalalignment='center',
#                          fontsize=12)
#         axes[idx, 3].axis('off')

#     plt.tight_layout()
#     plt.show()

# # Call the function
# display_samples_with_ground_truth(patches, labels, color_masks_)

In [ ]:
label_map = {'rough': 0, 'average': 1, 'smooth': 2, 'no_contour': 3}
numerical_labels = [label_map[label] for label in filtered_labels]

# Check class distribution
unique, counts = np.unique(filtered_labels, return_counts=True)
print("Class distribution:", dict(zip(unique, counts)))

# Data augmentation and preprocessing
transform_train = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Use weighted loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
weighted_criterion = create_weighted_loss_function(class_weights, device)

# Create balanced data loaders
train_loader, val_loader, label_map = create_balanced_data_loader(
    filtered_patches, filtered_labels, batch_size=32
)

In [ ]:
# Apply transforms to patches
# This part is redundant with create_balanced_data_loader, remove it
# X = []
# for patch in patches:
#     X.append(transform_train(patch))
# X = torch.stack(X)
# y = torch.LongTensor(numerical_labels)

# Split into train and validation sets (stratified)
# This part is redundant with create_balanced_data_loader, remove it
# X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# Create data loaders
# This part is redundant with create_balanced_data_loader, remove it
# train_dataset = TensorDataset(X_train, y_train)
# val_dataset = TensorDataset(X_val, y_val)
# train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_dataset, batch_size=32)

In [ ]:
# Use EfficientNet with proper weights parameter
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)
model = model.to(device)

# Define loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

Train the model with back tracking

In [9]:
! mkdir checkpoints
# Training loop with early stopping and checkpoint saving
num_epochs = 200
best_val_loss = float('inf')
patience = 5
trigger_times = 0
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
start_epoch = 0

# Check if a checkpoint exists
checkpoint_dir = "."
checkpoint_files = [f for f in os.listdir(checkpoint_dir) if f.startswith("checkpoint_epoch_")]
if checkpoint_files:
    latest_checkpoint = max(checkpoint_files, key=lambda x: int(x.split('_')[-1].split('.')[0]))
    checkpoint_path = os.path.join(checkpoint_dir, latest_checkpoint)
    checkpoint = torch.load(checkpoint_path, weights_only=True)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    start_epoch = checkpoint['epoch']
    best_val_loss = checkpoint['val_loss']
    train_losses = checkpoint.get('train_losses', [])
    val_losses = checkpoint.get('val_losses', [])
    train_accuracies = checkpoint.get('train_accuracies', [])
    val_accuracies = checkpoint.get('val_accuracies', [])
    print(f"Resuming training from epoch {start_epoch}")

for epoch in range(start_epoch, num_epochs):
    model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for inputs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs, 1)
        total_train += targets.size(0)
        correct_train += (predicted == targets).sum().item()

    epoch_train_loss = running_loss / len(train_dataset)
    train_accuracy = 100 * correct_train / total_train
    train_losses.append(epoch_train_loss)
    train_accuracies.append(train_accuracy)

    # Validation phase
    model.eval()
    val_loss = 0.0
    correct_val = 0
    total_val = 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_val += targets.size(0)
            correct_val += (predicted == targets).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    epoch_val_loss = val_loss / len(val_dataset)
    val_accuracy = 100 * correct_val / total_val
    val_losses.append(epoch_val_loss)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {epoch_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Val Loss: {epoch_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")

    # Save checkpoint
    checkpoint_path = f"checkpoints/checkpoint_epoch_{epoch+1}.pth"
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': epoch_train_loss,
        'val_loss': epoch_val_loss,
        'train_accuracy': train_accuracy,
        'val_accuracy': val_accuracy,
        'train_losses': train_losses,
        'val_losses': val_losses,
        'train_accuracies': train_accuracies,
        'val_accuracies': val_accuracies
    }, checkpoint_path)

    # Learning rate scheduling
    scheduler.step(epoch_val_loss)

    # Early stopping
    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        trigger_times = 0
        torch.save(model.state_dict(), "checkpoints/best_texture_model.pth")
    else:
        trigger_times += 1
        if trigger_times >= patience:
            print("Early stopping triggered!")
            break



NameError: name 'model' is not defined

In [ ]:
# Load the best model
model.load_state_dict(torch.load("checkpoints/best_texture_model.pth", map_location=device, weights_only=True))
# Compute additional metrics
precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='weighted')
print(f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}")

 confusion matrix and learning graph

In [8]:
# Confusion matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(8, 6))

# Use matplotlib's imshow to create a heatmap
plt.imshow(cm, interpolation='nearest', cmap='Blues')

# Add a colorbar
plt.colorbar()

# Add labels and ticks
tick_marks = np.arange(len(label_map))
plt.xticks(tick_marks, label_map.keys(), rotation=45)
plt.yticks(tick_marks, label_map.keys())

# Add text annotations to the heatmap
thresh = cm.max() / 2.  # Threshold for text color (for readability)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, format(cm[i, j], 'd'),
                 ha="center", va="center",
                 color="white" if cm[i, j] > thresh else "black")

plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

# Plot training graphs (unchanged)
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Over Epochs")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_accuracies, label="Train Acc")
plt.plot(val_accuracies, label="Val Acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Over Epochs")
plt.legend()
plt.tight_layout()
plt.show()

NameError: name 'all_targets' is not defined

for analysis show random sample for prediction and patches

In [7]:
def display_random_samples_with_predictions(model, original_images, patches, color_masks, label_map, fixed_indices=None, num_samples=20):
    model.eval()
    # Create reverse label map
    reverse_label_map = {v: k for k, v in label_map.items()}

    # Use fixed indices if provided, otherwise generate new ones
    if fixed_indices is None:
        indices = np.random.choice(len(patches), num_samples, replace=False)
    else:
        indices = fixed_indices

    for idx, sample_idx in enumerate(indices):
        # Create a new figure for each sample
        fig, axes = plt.subplots(1, 5, figsize=(20, 4))

        # Get patch and its label
        patch = patches[sample_idx]
        original_image = original_images[sample_idx]
        color_mask = color_masks[sample_idx]
        ground_truth = labels[sample_idx]

        # Extract patch color from color mask
        binary_cv = (binary_masks_[sample_idx] * 255).astype(np.uint8)
        contours, _ = cv2.findContours(binary_cv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest_contour = max(contours, key=cv2.contourArea)
            x, y, w, h = cv2.boundingRect(largest_contour)
            color_patch = cv2.resize(color_mask[y:y+h, x:x+w], (20, 20))
        else:
            color_patch = cv2.resize(color_mask, (20, 20))

        # Display original image
        axes[0].imshow(original_image)
        axes[0].set_title(f'Original image (idx:{sample_idx})')
        axes[0].axis('off')

        # Display patch
        axes[1].imshow(patch)
        axes[1].set_title('Patch image')
        axes[1].axis('off')

        # Display color mask
        axes[2].imshow(color_mask)
        axes[2].set_title('Color Mask')
        axes[2].axis('off')

        # Display color patch
        axes[3].imshow(color_patch)
        axes[3].set_title('Color Patch')
        axes[3].axis('off')

        # Get prediction
        patch_tensor = transform_val(patch).unsqueeze(0).to(device)
        with torch.no_grad():
            output = model(patch_tensor)
            _, predicted = torch.max(output, 1)

        # Convert numerical prediction to label name
        pred_label = reverse_label_map[predicted.item()]

        # Display labels
        axes[4].text(0.5, 0.5,
                     f'Predicted: {pred_label}\nGround Truth: {ground_truth}',
                     horizontalalignment='center',
                     verticalalignment='center',
                     fontsize=12)
        axes[4].axis('off')

        plt.tight_layout()
        plt.show()

    return indices


# Generate random indices once
fixed_sample_indices = [16, 900, 1110, 2217, 2302, 2566, 2854, 3613, 4872, 5648, 8159, 8307, 8531, 8971, 9031, 10443, 11053, 11696, 11863, 15064]

# # Call the function with fixed indices
display_random_samples_with_predictions(model, images_, patches, color_masks_, label_map, fixed_indices=fixed_sample_indices)

NameError: name 'model' is not defined

# Gen Code bottom

In [5]:
# -*- coding: utf-8 -*-
"""
Complete Roof Texture Classification System
Enhanced version with multiple rooftops per tile and improved training
"""

import os
import numpy as np
from PIL import Image
import cv2
from tqdm import tqdm
import torch
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, classification_report
from sklearn.utils.class_weight import compute_class_weight
import matplotlib.pyplot as plt
from torchvision import models, transforms
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision.models import EfficientNet_B0_Weights
from collections import Counter
import seaborn as sns

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# ======================== SETUP AND DATA LOADING ========================

def mount_drive_and_setup():
    """Setup Google Drive and create necessary directories"""
    from google.colab import drive
    drive.mount('/content/drive')

    # Check GPU and RAM
    gpu_info = !nvidia-smi
    gpu_info = '\n'.join(gpu_info)
    if gpu_info.find('failed') >= 0:
        print('❌ Not connected to a GPU')
    else:
        print('✅ GPU available')

    from psutil import virtual_memory
    ram_gb = virtual_memory().total / 1e9
    print(f'💾 RAM: {ram_gb:.1f} GB')

    # Setup paths
    base_path = '/content/drive/MyDrive/disser/data'

    # Create directories
    !mkdir -p /content/sample_data/data
    !mkdir -p /content/sample_data/data/tiles
    !mkdir -p /content/sample_data/data/texture_tiles
    !mkdir -p /content/sample_data/data/binary_tiles
    !mkdir -p /content/sample_data/data/patch
    !mkdir -p /content/sample_data/data/patch-binary
    !mkdir -p /content/sample_data/data/patch-texture
    !mkdir -p /content/sample_data/data/checkpoints

    return base_path

def copy_files_from_drive(base_path):
    """Copy files from Google Drive to local directory"""
    destination_folder = '/content/sample_data/data'

    for i in range(1, 13):
        source_tiles = os.path.join(base_path, f'tiles/{i}.jpg')
        source_texture = os.path.join(base_path, f'texture_tiles/{i}.png')
        source_binary = os.path.join(base_path, f'binary_tiles/{i}.png')

        !cp -f {source_tiles} {os.path.join(destination_folder, 'tiles')}
        !cp -f {source_binary} {os.path.join(destination_folder, 'binary_tiles')}
        !cp -f {source_texture} {os.path.join(destination_folder, 'texture_tiles')}

    print("✅ Files copied successfully")

# ======================== TILE GENERATION ========================

def get_all_char_before_dot(s):
    """Helper function to extract filename without extension"""
    dot_index = s.find('.')
    return s[:dot_index] if dot_index > 0 else None

def generate_tiles(image_path, tile_width, tile_height, step, output_folder):
    """Generate tiles from large images"""
    image = Image.open(image_path)
    image_width, image_height = image.size

    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            bbox = (x, y, x + tile_width, y + tile_height)
            tile = image.crop(bbox)

            image_file_name = os.path.basename(image_path)
            image_idx = get_all_char_before_dot(image_file_name)
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png")

def generate_binary_tiles(image_path, tile_width, tile_height, step, output_folder):
    """Generate binary tiles from mask images"""
    image = Image.open(image_path).convert("L")
    image_width, image_height = image.size

    for x in range(0, image_width - tile_width + 1, step):
        for y in range(0, image_height - tile_height + 1, step):
            bbox = (x, y, x + tile_width, y + tile_height)
            tile = image.crop(bbox)
            tile = tile.point(lambda p: 255 if p > 127 else 0, mode="1")

            image_file_name = os.path.basename(image_path)
            image_idx = image_file_name.split('.')[0]
            tile.save(f"{output_folder}/{image_idx}_{x}_{y}.png", format="PNG")

def generate_all_tiles(base_path):
    """Generate all tiles from original images"""
    # Generate tiles for original images
    for i in range(1, 13):
        generate_tiles(f'{base_path}/tiles/{i}.jpg', 256, 256, 128, f'{base_path}/patch')
    print("✅ Original tiles generated")

    # Generate binary tiles
    for i in range(1, 13):
        generate_binary_tiles(f'{base_path}/binary_tiles/{i}.png', 256, 256, 128, f'{base_path}/patch-binary')
    print("✅ Binary tiles generated")

    # Generate texture tiles
    for i in range(1, 13):
        generate_tiles(f'{base_path}/texture_tiles/{i}.png', 256, 256, 128, f'{base_path}/patch-texture')
    print("✅ Texture tiles generated")

# ======================== ENHANCED PATCH EXTRACTION ========================

def get_roof_texture_label(color_patch, min_pixel_threshold=50):
    """Enhanced function to extract roof texture label from color mask patch"""
    if len(color_patch.shape) == 3:
        image_rgb = color_patch
    else:
        image_rgb = cv2.cvtColor(color_patch, cv2.COLOR_BGR2RGB)

    red_channel = image_rgb[:, :, 0]
    green_channel = image_rgb[:, :, 1]
    blue_channel = image_rgb[:, :, 2]

    # Define thresholds
    high_threshold = 200
    low_threshold = 100

    # Count pixels with refined conditions
    rough_pixels = np.sum((red_channel > high_threshold) &
                         (green_channel < low_threshold) &
                         (blue_channel < low_threshold))

    smooth_pixels = np.sum((red_channel < low_threshold) &
                          (green_channel > high_threshold) &
                          (blue_channel < low_threshold))

    average_pixels = np.sum((red_channel > low_threshold) &
                           (red_channel < high_threshold) &
                           (green_channel > low_threshold) &
                           (green_channel < high_threshold) &
                           (blue_channel < low_threshold))

    pixel_counts = {
        "rough": rough_pixels,
        "smooth": smooth_pixels,
        "average": average_pixels
    }

    max_count = max(pixel_counts.values())

    if max_count < min_pixel_threshold:
        return "no_contour", pixel_counts
    else:
        label = max(pixel_counts.items(), key=lambda x: x[1])[0]
        return label, pixel_counts

def extract_multiple_rooftops_per_tile(image_paths, binary_paths, color_paths,
                                     min_contour_area=100, target_size=(224, 224)):
    """Extract multiple rooftops from each tile"""
    patches = []
    labels = []
    tile_info = []

    for idx in tqdm(range(len(binary_paths)), desc="Extracting patches"):
        try:
            # Load images
            orig_img = np.array(Image.open(image_paths[idx]).convert("RGB")) / 255.0
            binary_mask = np.array(Image.open(binary_paths[idx]).convert("L")) / 255.0
            color_mask = np.array(Image.open(color_paths[idx]).convert("RGB")) / 255.0

            # Convert binary mask for contour detection
            binary_cv = (binary_mask * 255).astype(np.uint8)

            # Find ALL contours
            contours, _ = cv2.findContours(binary_cv, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Filter and sort contours
            valid_contours = [c for c in contours if cv2.contourArea(c) > min_contour_area]

            if valid_contours:
                valid_contours.sort(key=cv2.contourArea, reverse=True)

                for contour_idx, contour in enumerate(valid_contours):
                    x, y, w, h = cv2.boundingRect(contour)

                    # Add padding
                    padding = 10
                    x = max(0, x - padding)
                    y = max(0, y - padding)
                    w = min(orig_img.shape[1] - x, w + 2*padding)
                    h = min(orig_img.shape[0] - y, h + 2*padding)

                    # Extract patches
                    orig_patch = orig_img[y:y+h, x:x+w]
                    color_patch = color_mask[y:y+h, x:x+w]

                    # Skip if too small
                    if orig_patch.shape[0] < 20 or orig_patch.shape[1] < 20:
                        continue

                    # Resize patches
                    orig_patch_resized = cv2.resize(orig_patch, target_size, interpolation=cv2.INTER_AREA)
                    color_patch_resized = cv2.resize(color_patch, target_size, interpolation=cv2.INTER_AREA)

                    # Get label
                    label, pixel_counts = get_roof_texture_label(
                        (color_patch_resized * 255).astype(np.uint8),
                        min_pixel_threshold=50
                    )

                    patches.append(orig_patch_resized)
                    labels.append(label)

                    # Store info
                    tile_info.append({
                        'tile_idx': idx,
                        'contour_idx': contour_idx,
                        'contour_area': cv2.contourArea(contour),
                        'bbox': (x, y, w, h),
                        'pixel_counts': pixel_counts,
                        'confidence': max(pixel_counts.values()) / sum(pixel_counts.values()) if sum(pixel_counts.values()) > 0 else 0
                    })
            else:
                # No valid contours
                orig_patch = cv2.resize(orig_img, target_size, interpolation=cv2.INTER_AREA)
                label = "no_contour"

                patches.append(orig_patch)
                labels.append(label)
                tile_info.append({
                    'tile_idx': idx,
                    'contour_idx': -1,
                    'contour_area': 0,
                    'bbox': (0, 0, orig_img.shape[1], orig_img.shape[0]),
                    'pixel_counts': {'rough': 0, 'smooth': 0, 'average': 0},
                    'confidence': 0
                })

        except Exception as e:
            print(f"Error processing tile {idx}: {str(e)}")
            continue

    return np.array(patches), labels, tile_info

def load_tiled_data(original_dir, binary_dir, color_dir):
    """Load paths for tiled data"""
    filenames = sorted(os.listdir(original_dir))
    image_paths, binary_paths, color_paths = [], [], []

    for filename in tqdm(filenames, desc="Loading tile paths"):
        img_path = os.path.join(original_dir, filename)
        binary_path = os.path.join(binary_dir, filename)
        color_path = os.path.join(color_dir, filename)

        if os.path.exists(img_path) and os.path.exists(binary_path) and os.path.exists(color_path):
            image_paths.append(img_path)
            binary_paths.append(binary_path)
            color_paths.append(color_path)

    return image_paths, binary_paths, color_paths

# ======================== DATA ANALYSIS AND FILTERING ========================

def analyze_data_distribution(labels, tile_info):
    """Analyze the distribution of extracted data"""
    print("\n=== DATA DISTRIBUTION ANALYSIS ===")

    # Count labels
    label_counts = Counter(labels)
    print(f"📊 Total patches extracted: {len(labels)}")
    print("Label distribution:")
    for label, count in label_counts.items():
        print(f"  {label}: {count} ({count/len(labels)*100:.1f}%)")

    # Analyze confidence scores
    confidences = [info['confidence'] for info in tile_info]
    print(f"\n🎯 Confidence scores:")
    print(f"  Mean: {np.mean(confidences):.3f}")
    print(f"  Median: {np.median(confidences):.3f}")
    print(f"  Min: {np.min(confidences):.3f}")
    print(f"  Max: {np.max(confidences):.3f}")

    # Count patches per tile
    tiles_with_patches = Counter([info['tile_idx'] for info in tile_info])
    patches_per_tile = list(tiles_with_patches.values())
    print(f"\n🔢 Patches per tile:")
    print(f"  Mean: {np.mean(patches_per_tile):.1f}")
    print(f"  Median: {np.median(patches_per_tile):.1f}")
    print(f"  Max: {np.max(patches_per_tile)}")

    return label_counts, confidences, patches_per_tile

def filter_low_confidence_samples(patches, labels, tile_info, min_confidence=0.3):
    """Filter out low confidence samples"""
    print(f"\n🔍 Filtering samples with confidence < {min_confidence}")

    filtered_patches = []
    filtered_labels = []
    filtered_info = []

    for patch, label, info in zip(patches, labels, tile_info):
        if info['confidence'] >= min_confidence or label == "no_contour":
            filtered_patches.append(patch)
            filtered_labels.append(label)
            filtered_info.append(info)

    print(f"✅ Kept {len(filtered_patches)} out of {len(patches)} samples")
    return np.array(filtered_patches), filtered_labels, filtered_info

def visualize_extracted_patches(patches, labels, tile_info, num_samples=16):
    """Visualize sample patches"""
    indices = np.random.choice(len(patches), min(num_samples, len(patches)), replace=False)

    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    axes = axes.flatten()

    for i, idx in enumerate(indices):
        patch = patches[idx]
        label = labels[idx]
        confidence = tile_info[idx]['confidence']

        axes[i].imshow(patch)
        axes[i].set_title(f'{label}\nConf: {confidence:.2f}', fontsize=10)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# ======================== MODEL TRAINING SETUP ========================

def compute_class_weights(labels):
    """Compute class weights for balanced training"""
    unique_labels = list(set(labels))
    label_to_idx = {label: idx for idx, label in enumerate(unique_labels)}
    numerical_labels = [label_to_idx[label] for label in labels]

    class_weights = compute_class_weight('balanced',
                                       classes=np.unique(numerical_labels),
                                       y=numerical_labels)

    print("\n⚖️ Class weights:")
    for label, idx in label_to_idx.items():
        print(f"  {label}: {class_weights[idx]:.3f}")

    return torch.FloatTensor(class_weights), label_to_idx

def create_data_loaders(patches, labels, label_to_idx, batch_size=32, test_size=0.2):
    """Create balanced data loaders"""
    # Data transformations
    transform_train = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    transform_val = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Convert labels to numerical
    numerical_labels = [label_to_idx[label] for label in labels]

    # Apply transforms
    X_train_transformed = []
    X_val_transformed = []

    # Convert to tensors first
    X = torch.stack([torch.from_numpy(patch.transpose(2, 0, 1)).float() for patch in patches])
    y = torch.LongTensor(numerical_labels)

    # Stratified split
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    # Apply transforms after split
    for patch in X_train:
        patch_img = patch.permute(1, 2, 0).numpy()
        X_train_transformed.append(transform_train(patch_img))

    for patch in X_val:
        patch_img = patch.permute(1, 2, 0).numpy()
        X_val_transformed.append(transform_val(patch_img))

    X_train = torch.stack(X_train_transformed)
    X_val = torch.stack(X_val_transformed)

    # Create weighted sampler for training
    class_sample_count = np.array([len(np.where(y_train == t)[0]) for t in np.unique(y_train)])
    weight = 1. / class_sample_count
    samples_weight = np.array([weight[t] for t in y_train])
    samples_weight = torch.from_numpy(samples_weight).double()
    sampler = WeightedRandomSampler(samples_weight, len(samples_weight))

    # Create datasets and loaders
    train_dataset = TensorDataset(X_train, y_train)
    val_dataset = TensorDataset(X_val, y_val)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader

def create_model(num_classes, class_weights, device):
    """Create and setup the model"""
    model = models.efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    model = model.to(device)

    # Weighted loss function
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)

    return model, criterion, optimizer, scheduler

# ======================== TRAINING LOOP ========================

def train_model(model, criterion, optimizer, scheduler, train_loader, val_loader,
                num_epochs=100, patience=10, checkpoint_dir="/content/sample_data/data/checkpoints"):
    """Train the model with early stopping"""

    device = next(model.parameters()).device
    best_val_loss = float('inf')
    trigger_times = 0
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    print(f"\n🚀 Starting training for {num_epochs} epochs")
    print(f"📱 Using device: {device}")

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for inputs, targets in train_bar:
            inputs, targets = inputs.to(device), targets.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += targets.size(0)
            correct_train += (predicted == targets).sum().item()

            train_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100 * correct_train / total_train:.2f}%'
            })

        epoch_train_loss = running_loss / len(train_loader.dataset)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(epoch_train_loss)
        train_accuracies.append(train_accuracy)

        # Validation phase
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0
        all_preds, all_targets = [], []

        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]")
            for inputs, targets in val_bar:
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total_val += targets.size(0)
                correct_val += (predicted == targets).sum().item()

                all_preds.extend(predicted.cpu().numpy())
                all_targets.extend(targets.cpu().numpy())

                val_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100 * correct_val / total_val:.2f}%'
                })

        epoch_val_loss = val_loss / len(val_loader.dataset)
        val_accuracy = 100 * correct_val / total_val
        val_losses.append(epoch_val_loss)
        val_accuracies.append(val_accuracy)

        print(f"Epoch {epoch+1}: Train Loss: {epoch_train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {val_accuracy:.2f}%")

        # Save checkpoint
        checkpoint_path = f"{checkpoint_dir}/checkpoint_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': epoch_train_loss,
            'val_loss': epoch_val_loss,
            'train_accuracy': train_accuracy,
            'val_accuracy': val_accuracy,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'train_accuracies': train_accuracies,
            'val_accuracies': val_accuracies
        }, checkpoint_path)

        # Learning rate scheduling
        scheduler.step(epoch_val_loss)

        # Early stopping
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            trigger_times = 0
            torch.save(model.state_dict(), f"{checkpoint_dir}/best_texture_model.pth")
            print(f"✅ New best model saved with val_loss: {best_val_loss:.4f}")
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f"🛑 Early stopping triggered after {patience} epochs without improvement!")
                break

    return train_losses, val_losses, train_accuracies, val_accuracies, all_preds, all_targets

# ======================== EVALUATION AND VISUALIZATION ========================

def evaluate_model(all_targets, all_preds, label_to_idx):
    """Evaluate model performance"""
    # Reverse label mapping
    idx_to_label = {v: k for k, v in label_to_idx.items()}

    # Calculate metrics
    precision, recall, f1, _ = precision_recall_fscore_support(all_targets, all_preds, average='weighted')

    print(f"\n📊 FINAL RESULTS:")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1-Score: {f1:.4f}")

    # Classification report
    print(f"\n📋 Classification Report:")
    target_names = [idx_to_label[i] for i in sorted(idx_to_label.keys())]
    print(classification_report(all_targets, all_preds, target_names=target_names))

    return precision, recall, f1

def plot_training_history(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training history"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    ax1.plot(train_losses, label="Train Loss", color='blue')
    ax1.plot(val_losses, label="Val Loss", color='red')
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss Over Epochs")
    ax1.legend()
    ax1.grid(True)

    # Accuracy plot
    ax2.plot(train_accuracies, label="Train Acc", color='blue')
    ax2.plot(val_accuracies, label="Val Acc", color='red')
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy (%)")
    ax2.set_title("Accuracy Over Epochs")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(all_targets, all_preds, label_to_idx):
    """Plot confusion matrix"""
    idx_to_label = {v: k for k, v in label_to_idx.items()}
    labels = [idx_to_label[i] for i in sorted(idx_to_label.keys())]

    cm = confusion_matrix(all_targets, all_preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()
